# **Système Multi-Agent SAV : Orchestrateur LangGraph + Serveur MCP (`rag_multimodal`)**
**Auteur :** Walid IDIR

Ce notebook implémente et exécute l'architecture SAV complète :
- **Orchestrateur LangGraph (`orchestrateur`)** : analyse la demande, sélectionne un seul agent spécialisé (`agent_documentaire` ou `agent_garantie`), puis termine dès qu'une réponse finale est disponible.
- **`agent_documentaire`** : consomme l'outil distant **`rag_multimodal`** exposé par le serveur MCP ([sav_mcp_server.py](file:///c:/Users/user/Desktop/TP%20LANG/sav_mcp_server.py)) via `streamable-http`.
- **`agent_garantie`** : interroge l'outil local **`consulter_garantie`** branché sur un registre en lecture seule (`MappingProxyType`).


In [1]:
import asyncio
import socket
import subprocess
import sys
import time
from IPython.display import Markdown, display
from langchain_core.messages import HumanMessage

from sav_orchestrator import graph, consulter_garantie, REGISTRE_GARANTIE

print("✅ Module `sav_orchestrator` importé avec succès.")
print("Numéros de série en registre (lecture seule) :", list(REGISTRE_GARANTIE.keys()))


✅ Module `sav_orchestrator` importé avec succès.
Numéros de série en registre (lecture seule) : ['PX200-0042', 'PX200-0011', 'RX500-0108']


## **1. Topologie du graphe LangGraph (`orchestrateur` ↔ agents spécialisés)**


In [2]:
mermaid_diagram = graph.get_graph().draw_mermaid()
print(mermaid_diagram)


---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	orchestrateur(orchestrateur)
	agent_documentaire(agent_documentaire)
	agent_garantie(agent_garantie)
	__end__([<p>__end__</p>]):::last
	__start__ --> orchestrateur;
	agent_documentaire --> orchestrateur;
	agent_garantie --> orchestrateur;
	orchestrateur -.-> __end__;
	orchestrateur -.-> agent_documentaire;
	orchestrateur -.-> agent_garantie;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc


## **2. Démarrage du serveur MCP SAV (`rag_multimodal`) et exécution des deux exemples**


In [3]:
# Démarre sav_mcp_server.py si le port 24001 n'est pas déjà actif
started_here = None
try:
    with socket.create_connection(("127.0.0.1", 24001), timeout=1):
        print("ℹ️ Serveur MCP SAV déjà actif sur le port 24001.")
except OSError:
    started_here = subprocess.Popen(
        [sys.executable, "sav_mcp_server.py"],
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
    )
    for _ in range(25):
        time.sleep(1)
        try:
            with socket.create_connection(("127.0.0.1", 24001), timeout=1):
                print("🚀 Serveur MCP SAV démarré sur http://127.0.0.1:24001/mcp")
                break
        except OSError:
            pass

try:
    # Exemple 1 — Assistance documentaire
    ex1 = asyncio.run(
        graph.ainvoke({
            "messages": [
                HumanMessage(
                    content="Mon imprimante PX-200 affiche un voyant clignotant. "
                    "Que signifie le pictogramme sur la photo jointe ?"
                )
            ],
            "image_ref": "img_px200_01",
        })
    )
    print(f"\n=== EXEMPLE 1 ===\nCatégorie : {ex1['categorie']} | Agent : {ex1['agent_selectionne']}")
    display(Markdown(ex1["reponse_finale"]))

    # Exemple 2 — Vérification de garantie
    ex2 = asyncio.run(
        graph.ainvoke({
            "messages": [
                HumanMessage(
                    content="Est-ce que mon imprimante est encore sous garantie ? "
                    "Son numéro de série est PX200-0042."
                )
            ]
        })
    )
    print(f"\n=== EXEMPLE 2 ===\nCatégorie : {ex2['categorie']} | Agent : {ex2['agent_selectionne']}")
    display(Markdown(ex2["reponse_finale"]))
finally:
    if started_here is not None:
        subprocess.run(["taskkill", "/F", "/T", "/PID", str(started_here.pid)], capture_output=True)
        print("\n🛑 Serveur MCP SAV temporaire arrêté.")


🚀 Serveur MCP SAV démarré sur http://127.0.0.1:24001/mcp
🧭 [Orchestrateur] Catégorie='documentation' | Agent sélectionné='agent_documentaire'
📘 [agent_documentaire] Traitement de la demande documentaire (image=img_px200_01)...
🌐 [Client MCP -> http://127.0.0.1:24001/mcp] Appel de `rag_multimodal` avec {'question': 'Mon imprimante PX-200 affiche un voyant clignotant. Que signifie le pictogramme sur la photo jointe ?', 'image_ref': 'img_px200_01'}
🏁 [Orchestrateur] Réponse finale disponible -> fin du workflow (END).
=== EXEMPLE 1 ===
Catégorie : documentation | Agent : agent_documentaire
🧭 [Orchestrateur] Catégorie='garantie' | Agent sélectionné='agent_garantie'
🛡️ [agent_garantie] Traitement de la demande de vérification de garantie...
🛠️ [Tool Local] consulter_garantie(numero_serie='PX200-0042')
🏁 [Orchestrateur] Réponse finale disponible -> fin du workflow (END).
=== EXEMPLE 2 ===
Catégorie : garantie | Agent : agent_garantie
🛑 Serveur MCP SAV temporaire arrêté.


D'après le manuel technique **Manuel_PX200_v2.pdf** (page 18, figure **FIG-4.2**) :

[Manuel PX-200 | Section 4.2 — Voyants et pictogrammes d'état | Page 18 | Figure FIG-4.2]
Voyant orange clignotant accompagné du pictogramme d'une feuille avec flèche courbée (référence visuelle : img_px200_01) : indique un bourrage papier dans le bac d'alimentation arrière ou un capteur de passage papier obstrué. Action recommandée : éteindre l'appareil, ouvrir le capot arrière, retirer délicatement la feuille coincée sans la déchirer, puis appuyer 2 secondes sur Reset.

**Sources consultées :** Manuel_PX200_v2.pdf (p. 18, FIG-4.2), Manuel_RX500_v1.pdf (p. 12, FIG-3.1)


Vérification de garantie pour le numéro de série **PX200-0042** (Imprimante PX-200) :
- **Statut :** Sous garantie
- **Date d'achat :** 2025-06-15
- **Fin de garantie :** 2027-06-15
- **Couverture :** Garantie constructeur 2 ans (pièces et main-d'œuvre)
